## Setup (run every session)
Large artifacts go to Google Drive (`PA3_OUT`); code and small results live in the repo clone. Use the **same GPU type** for every timed run and keep the tab open (Colab Pro has no background execution).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['PA3_OUT'] = '/content/drive/MyDrive/ATML-PA3/outputs'   # checkpoints, caches (persist)
os.environ['PA3_COCO'] = '/content/coco'                             # images on fast local disk
os.environ['HF_HOME'] = '/content/hf_cache'
REPO = 'https://github.com/adilawan1/ATML-PA3.git'                 # <-- set


In [ ]:
%cd /content
!test -d ATML-PA3 || git clone $REPO ATML-PA3
%cd /content/ATML-PA3
!git pull -q
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Task 1: LLM fine-tuning
### 1. Data (deterministic; recomputes the committed selection and checks its hash)

In [ ]:
!python -m task1.prepare_data
!git diff --stat results/task1/data_selection.json   # should show no change

### 2. Training: one cell per variant. Re-running a cell resumes from the newest checkpoint.

In [ ]:
!python -m task1.train --variant full_ft

In [ ]:
!python -m task1.train --variant full_ft_l2sp

In [ ]:
!python -m task1.train --variant lora4

In [ ]:
!python -m task1.train --variant lora16

In [ ]:
!python -m task1.train --variant oplora16

### 3. Export (merge + logit check), benchmark evaluation, qualitative generations for the 7 conditions
Merged FP16 models are written to local disk and deleted after use. Evaluate the baseline first to validate the lm-eval setup.

In [ ]:
import subprocess, shutil
CONDS = ['baseline', 'full_ft_mid', 'full_ft', 'full_ft_l2sp', 'lora4', 'lora16', 'oplora16']
for c in CONDS:
    if os.path.exists(f'results/task1/eval/{c}/scores.json') and os.path.exists(f'results/task1/qualitative/{c}.json'):
        print('skip', c); continue
    subprocess.run(['python', '-m', 'task1.export_models', '--condition', c, '--out_dir', '/content/merged'], check=True)
    subprocess.run(['python', '-m', 'task1.evaluate', '--condition', c, '--model_dir', f'/content/merged/{c}'], check=True)
    subprocess.run(['python', '-m', 'task1.qualitative', '--condition', c, '--model_dir', f'/content/merged/{c}'], check=True)
    shutil.rmtree(f'/content/merged/{c}')

### 4. Diagnostics, qualitative collection, tables and figures

In [ ]:
!python -m task1.diagnostics
!python -m task1.qualitative --collect
!python -m task1.analyze

Annotate `results/task1/qualitative/annotation_sheet.csv` (42 rows) by hand: fill the `ann_*` columns.

## Save small results back to the repo
Commit from Colab (needs a token) or download `results/` and commit locally.

In [ ]:
!git add results && git -c user.name=pa3 -c user.email=pa3@example.com commit -qm "results" || true
# !git push https://<token>@github.com/adilawan1/ATML-PA3.git HEAD